# Imports

In [16]:
import pandas as pd
import sys
import os
import plotly.express as px
from tqdm import tqdm
import numpy as np
sys.path.append('..')
from src import helpers
from scipy.sparse import csgraph, csr_matrix
from definitions import DATA_DIR, ROOT_DIR
from functools import reduce
from plotly.graph_objs import Figure
import plotly.io as pio
from functools import reduce

In [3]:
FIGURES_DIR = os.path.join(ROOT_DIR, 'assets', 'graph_statistics')
os.makedirs(FIGURES_DIR, exist_ok=True)

In [4]:
pio.templates.default = "seaborn"

In [5]:
def save_fig(fig : Figure, filename : str) -> None:
    fig.write_image(os.path.join(FIGURES_DIR, filename))

In [6]:
def show(fig : Figure) -> None:
    fig.update_layout(font_size = 12)
    fig.show()

# Utils

In [13]:
def load_graph(
    corpus : str,
    predicate_min_occ : int = 30,
    entity_min_occ : int = 2
) -> pd.DataFrame: 
    
    path = os.path.join(DATA_DIR, corpus, 'clean_triples', 'data.jsonl')
    kg = pd.read_json(path, lines=True)

    predicates = kg['predicate'].value_counts()
    entities = pd.Series(data = kg['head'].to_list() + kg['tail'].to_list()).value_counts()

    predicates = predicates.index[predicates >= predicate_min_occ]
    entities = entities.index[entities >= entity_min_occ]

    mask = (
        kg['head'].isin(entities) &
        kg['tail'].isin(entities) &
        kg['predicate'].isin(predicates)
    )

    return kg[mask]

In [21]:
def melt_df(kg : pd.DataFrame) -> pd.DataFrame:

    melted = []

    for i,(_,row) in tqdm(enumerate(kg.iterrows()), total=len(kg)):
                        
        for key in ["head","tail"]:

            melted.append({
                "triple_id" : 2 * i + int(key == "head"),
                "entity" : row[key],
                "predicate" : row["predicate"],
                "head" : key == "head",
                "tail" : key == "tail",
                "document" : row['sentence']['document']['id']
            })

    melted = pd.DataFrame(melted)

    return melted

In [17]:
def get_components(df : pd.DataFrame) -> pd.Series:

    entities = list(set(df['head'].to_list() + df['tail'].to_list()))
    entities_map = { e : i for i,e in enumerate(entities) }
    head = df['head'].map(entities_map).to_numpy()
    tail = df['tail'].map(entities_map).to_numpy()

    N = len(entities)
    indices = np.stack([head, tail]).transpose()
    data = np.ones(len(df))

    adj = csr_matrix(
        (data, (indices[:, 0], indices[:, 1])),
        shape=(N, N),
        dtype=int
    )

    _, labels = csgraph.connected_components(adj, directed=False)

    labels = pd.Series(
        index=entities,
        data=labels
    )

    return labels

In [22]:
def describe_graph(kg : pd.DataFrame) -> tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.Series]:

    ### Metl-Graph
    melted = melt_df(kg)

    ### Node-Level
    print("Node level")
    node_level : list[pd.Series] = [
        melted['document'].groupby(melted['entity']).agg(lambda x : len(set(x))).rename('number-of-documents'),
    ]

    melted = melted.drop_duplicates(subset=["triple_id"])

    node_level.extend([
        melted['head'].groupby(melted['entity']).sum().rename('out-degree'),
        melted['tail'].groupby(melted['entity']).sum().rename('in-degree'),
        melted['entity'].value_counts().rename('degree'),
    ])

    node_level = reduce(
        lambda a,b : a.join(b.to_frame()),
        node_level,
        pd.DataFrame(index = node_level[0].index)
    )

    ### Relation-Level
    print("Predicate Level")
    relation_level : list[pd.Series] = [
        kg['predicate'].value_counts().rename('number-of-triples')
    ]
    
    relation_level = reduce(
        lambda a,b : a.join(b.to_frame()),
        relation_level,
        pd.DataFrame(index = relation_level[0].index)
    )

    ### Document Level
    print("Document Level")
    document_level : list[pd.Series] = [
        melted['document'].value_counts().rename('number-of-triples'),
        melted['entity'].groupby(by = melted['document']).agg(lambda x : len(set(x))).rename('number-of-entities')
    ]

    document_level : list[pd.Series] = reduce(
        lambda a,b : a.join(b.to_frame()),
        document_level,
        pd.DataFrame(index = document_level[0].index)
    )
    

    ### Components
    print("Connected Componenets")
    components = get_components(kg)
    component_level = [
        components.value_counts().rename('number-of-entities'),
        kg['head'].map(components).value_counts().rename('number-of-triples')
    ]

    component_level = reduce(
        lambda a,b : a.join(b.to_frame()),
        component_level,
        pd.DataFrame(index = component_level[0].index)
    )

    ### Graph-Level
    print("Graph Level")
    graph_level = dict()

    graph_level['Number of Triples'] = len(kg)
    graph_level['Number of Entities'] = melted['entity'].nunique()
    graph_level['Number of Predicates'] = melted['predicate'].nunique()

    graph_level['Avg. In-Degree'] = node_level['in-degree'].mean()
    graph_level['Avg. Out-Degree'] = node_level['out-degree'].mean()
    graph_level['Avg. Degree'] = node_level['degree'].mean()
    graph_level['Avg. Documents / Entity'] = node_level['number-of-documents'].mean()

    graph_level['Avg. Triples / Predicate'] = relation_level['number-of-triples'].mean()

    graph_level['Avg. Triples / Document'] = document_level['number-of-triples'].mean()
    graph_level['Avg. Entity / Document'] = document_level['number-of-entities'].mean()

    graph_level['Largest Connected Component (Number of Entities)'] = component_level['number-of-entities'].max()
    graph_level['Largest Connected Component (Number of Triples)'] = component_level['number-of-triples'].max()
    graph_level['Number Of Connected Components'] = len(component_level)

    graph_level = pd.Series(graph_level)

    return node_level, relation_level, document_level, component_level, graph_level

# Load the data

In [19]:
kg = load_graph('ls2n-duke', entity_min_occ = 1, predicate_min_occ = 1)
kg.head()

,id,sentence,head,predicate,tail,predicate_group
0,547194300,"{'id': '5471943_0', 'sentence': 'Accurately es...",latency,estimate,software applications,estimating
1,547194301,"{'id': '5471943_0', 'sentence': 'Accurately es...",power consumption,estimate,software applications,estimating
2,547194302,"{'id': '5471943_0', 'sentence': 'Accurately es...",software applications,deploy on,multicore systems,deployed
3,547194310,"{'id': '5471943_1', 'sentence': 'This paper ex...",modeling application to architecture mappings,explore as,heterogeneous graphs,explores as
4,547194320,"{'id': '5471943_2', 'sentence': 'Four GNN mode...",gnn models,evaluate across,datasets,evaluated across


In [23]:
node_level, relation_level, document_level, component_level, graph_level = describe_graph(kg)

100%|██████████| 1652/1652 [00:00<00:00, 31878.53it/s]

Node level
Predicate Level
Document Level
Connected Componenets
Graph Level


In [25]:
graph_level

Number of Triples                                   1652.000000
Number of Entities                                  1661.000000
Number of Predicates                                 571.000000
Avg. In-Degree                                         0.994582
Avg. Out-Degree                                        0.994582
Avg. Degree                                            1.989163
Avg. Documents / Entity                                1.166767
Avg. Triples / Predicate                               2.893170
Avg. Triples / Document                               28.239316
Avg. Entity / Document                                16.564103
Largest Connected Component (Number of Entities)     499.000000
Largest Connected Component (Number of Triples)      619.000000
Number Of Connected Components                       266.000000
dtype: float64

# Visualization

## Node-Level 

- In-Degrees

In [26]:
fig = px.histogram(
    data_frame = node_level,
    x = "in-degree",
    width = 800,
    log_y=True
)

fig.update_layout(xaxis_title = "In-Degree", yaxis_title = "Number of Entities", font_size=17)

save_fig(fig, 'in_degree_distribution.svg')
show(fig)

- Out Degree

In [27]:
fig = px.histogram(
    data_frame = node_level,
    x = "out-degree",
    width = 800,
    log_y=True
)
fig.update_layout(xaxis_title = "Out-Degree", yaxis_title = "Number of Entities")
save_fig(fig, 'out_degree_distribution.svg')
show(fig)

- Degree

In [28]:
fig = px.histogram(
    data_frame = node_level,
    x = "degree",
    width = 800,
    log_y=True
)

fig.update_layout(xaxis_title = "Degree", yaxis_title = "Number of Entities")
save_fig(fig, 'node_degree_distribution.svg')
show(fig)

- Most frequent entities

In [29]:
fig = px.bar(
    data_frame = node_level['degree'].sort_values(ascending = False).reset_index()[:20][::-1],
    y = "entity",
    x = "degree",
    width = 800,
)

fig.update_layout(xaxis_title = "Number of Triplets", yaxis_title = "Entity")
save_fig(fig, 'top_10_most_frequent_entities.svg')
show(fig)

- Predicates

In [30]:
fig = px.histogram(
    data_frame = relation_level.reset_index(),
    x = "number-of-triples",
    width = 800,
    log_y=True
)
fig.update_layout(xaxis_title = 'Relation Frequency', yaxis_title = "Number Of Relations")
save_fig(fig, 'relations_frequency_distribution.svg')
show(fig)

In [31]:
fig = px.bar(
    data_frame = relation_level.sort_values(by = "number-of-triples", ascending = False).reset_index()[::20][::-1],
    y = "predicate",
    x = "number-of-triples",
    width = 800,
    log_x=True
)
fig.update_layout(xaxis_title = 'Number of Triplets', yaxis_title = "Predicate")
save_fig(fig, 'top_10_most_frequent_predicates.svg')
show(fig)

- Graph Connectivity

In [32]:
fig = px.histogram(
    data_frame = component_level,
    x = 'number-of-entities',
    width = 800,
    log_y = True
)
fig.update_layout(xaxis_title = 'Subgraph Size (Number of Nodes)', yaxis_title = "Number of Subgraphs")
save_fig(fig, 'subraphs_distribution.svg')
show(fig)

In [33]:
fig = px.bar(
    data_frame = component_level.iloc[:10].reset_index(),
    x = 'index', 
    y = 'number-of-entities',
    width = 800,
    log_y = True
)
fig.update_layout(xaxis_title = 'Subgraph-ID', yaxis_title = "Subgraph Size (Number of Nodes)")
save_fig(fig, 'top_10_largest_subgraphs.svg')
show(fig)